# Zero-shot prompt routing with aibackends + LFM2.5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/aibackends/blob/main/examples/notebooks/lfm25_prompt_routing_colab.ipynb)

This notebook demos the routing backend built on
[LFM2.5-Encoder-350M-Prompt-Router](https://huggingface.co/LiquidAI/LFM2.5-Encoder-350M-Prompt-Router),
a 350M-parameter bidirectional encoder fine-tuned by Liquid AI for zero-shot prompt routing.

Routing lanes are **free text you make up at call time** — no fixed taxonomy, no classifier
training. The encoder reads the whole prompt in one forward pass and scores it against every
lane at once. It runs **entirely on your machine** through `transformers`
(`trust_remote_code=True`; the routing head lives in the model repo's custom code), and a warm
call takes roughly a quarter second on CPU.

Like GliGuard moderation, it does *not* go through your configured generative runtime — it is
an independent capability backend, so routing stays fast and offline no matter which LLM you
serve behind it.

**What this notebook covers**

1. Load once, reuse everywhere (cold vs. warm cost)
2. Device-assistant routing — simple tool calls vs. complex agentic work
3. Adding a category on the fly (a "soccer agent")
4. Code-language routing as a batch table
5. Threshold tuning
6. Multilingual routing
7. Routing to model tiers by complexity (local models vs. frontier cloud models)
8. Async calls
9. Doing the same from the CLI

> Runs fine on a **free CPU runtime**. A GPU runtime works too and is picked up automatically.

## Setup

The `routing` extra pulls in `torch` and `transformers`.

In [ ]:
!pip install -q "aibackends[routing]"

# If a later import fails with a transformers error, use
# Runtime > Restart session, then continue from the next cell.

In [ ]:
import time

import aibackends
from aibackends.backends.routing import get_routing_backend, list_routing_backends

print('aibackends', aibackends.__version__)
print('routing backends:', list_routing_backends())

# The router accepts: cpu, gpu, cuda, cuda:<index>, mps
try:
    import torch
    DEVICE = 'gpu' if torch.cuda.is_available() else 'cpu'
except ImportError:
    DEVICE = 'cpu'

print('device:', DEVICE)

## 1. Load once, reuse everywhere

The model is cached per process **and** per device, so you pay the load cost once.
The first run of the cell below also downloads ~1.4GB of weights from the Hugging Face Hub.

In [ ]:
backend = get_routing_backend('lfm2-prompt-router')

t = time.perf_counter()
backend.load(device=DEVICE)
load_s = time.perf_counter() - t
print(f'model load: {load_s:.1f}s')

## 2. Device-assistant routing

The lanes below mirror Liquid AI's demo: an on-device assistant that separates cheap
function calls from work that needs a real agent (or a bigger model).

`route_prompt` returns a `RoutingResult`:

| Field | Meaning |
|---|---|
| `best_route` | highest-scoring lane (or `None` if a threshold filtered everything) |
| `scores` | every lane with its softmax score, sorted descending |
| `backend_used` / `model_id` | provenance for logging |

In [ ]:
from aibackends.tasks import route_prompt

ASSISTANT_LANES = [
    'Simple function call',
    'Simple tool use',
    'Complex multi-step agentic task',
    'Quick factual question',
    'Casual conversation',
    'Creative writing',
    'Translation',
    'Needs a bigger model',
]


def show(result, top=4):
    print(f'prompt: {result.text}')
    for score in result.scores[:top]:
        marker = '->' if score.route == result.best_route else '  '
        print(f'  {marker} {score.route:<34} {score.score:6.1%}')
    print()


t = time.perf_counter()
easy = route_prompt(
    "What's the temperature in San Francisco?", ASSISTANT_LANES, device=DEVICE
)
warm_ms = (time.perf_counter() - t) * 1000

hard = route_prompt(
    'Plan and book a three-day trip to Tokyo with hotels under $200 a night.',
    ASSISTANT_LANES,
    device=DEVICE,
)

show(easy)
show(hard)
print(f'warm call: {warm_ms:.0f}ms  (vs {load_s * 1000:.0f}ms to load)')

## 3. Adding a category on the fly

There is no training step, so extending the taxonomy is just appending a string.
Watch a soccer question land in a generic lane, then claim its own agent once the
lane exists.

In [ ]:
question = 'Who won the 2026 FIFA World Cup?'

before = route_prompt(question, ASSISTANT_LANES, device=DEVICE)
after = route_prompt(question, [*ASSISTANT_LANES, 'Soccer agent'], device=DEVICE)

print(f'without the lane: {before.best_route!r} ({before.scores[0].score:.1%})')
print(f'with the lane:    {after.best_route!r} ({after.scores[0].score:.1%})')

## 4. Code-language routing as a batch table

Domain-specific routing: send each bug report to the right language expert.
`route_prompts` reuses the loaded model across the whole batch.

In [ ]:
import pandas as pd

from aibackends.tasks import route_prompts

LANGUAGE_LANES = [
    'Python', 'JavaScript', 'TypeScript', 'Go', 'Rust', 'Java', 'C++', 'PHP', 'Ruby',
]

bug_reports = [
    'np.einsum returns the wrong shape when broadcasting over the batch axis.',
    'The borrow checker rejects the lifetime in my iterator adapter.',
    'How do I narrow a union type inside a switch statement?',
    'My Laravel migration fails with a foreign key constraint error.',
    'Goroutines leak when the context is cancelled before the send.',
]

t = time.perf_counter()
results = route_prompts(bug_reports, LANGUAGE_LANES, device=DEVICE)
batch_s = time.perf_counter() - t
per_item_ms = batch_s / len(bug_reports) * 1000

table = pd.DataFrame(
    {
        'prompt': [p[:48] + ('...' if len(p) > 48 else '') for p in bug_reports],
        'lane': [r.best_route for r in results],
        'confidence': [f'{r.scores[0].score:.1%}' for r in results],
    }
)

print(f'{len(bug_reports)} prompts in {batch_s:.2f}s  ({per_item_ms:.0f}ms each)\n')
table

## 5. Threshold tuning

Scores are a softmax over your lanes, so they always sum to 1. Passing `threshold`
drops lanes below the bar — useful as an "unsure, escalate to a human" switch:
when nothing clears it, `best_route` comes back `None`.

In [ ]:
ambiguous = 'Can you do something about my account?'

for threshold in (None, 0.3, 0.6):
    result = route_prompt(
        ambiguous,
        ['billing question', 'account access problem', 'bug report'],
        device=DEVICE,
        threshold=threshold,
    )
    lanes = [(s.route, round(s.score, 2)) for s in result.scores]
    print(f'threshold={threshold!s:5} best={result.best_route!r} kept={lanes}')

## 6. Multilingual routing

The encoder is trained on 15 languages, so the same English lanes catch prompts
written in German, Spanish, or Japanese.

In [ ]:
INTENT_LANES = ['billing question', 'bug report', 'feature request', 'account access problem']

multilingual = [
    'I was charged twice for my subscription this month.',
    'Die App stuerzt ab, sobald ich den Export-Button klicke.',
    'No puedo iniciar sesion, mi cuenta parece bloqueada.',
    'ダークモードを追加してもらえますか？',
]

for r in route_prompts(multilingual, INTENT_LANES, device=DEVICE):
    print(f'{r.best_route:<24} {r.scores[0].score:6.1%}  {r.text}')

## 7. Routing to model tiers by complexity

The pattern from the LFM2.5-Encoders release: a small encoder makes an inexpensive
first pass so that cheap models absorb the easy traffic and frontier models only see
the prompts that need them.

aibackends ships **local runtimes only** (llama.cpp and transformers), so the cloud
lanes below print the chosen provider and model id — that hand-off is where you would
call the provider SDK. The local lanes are real: `lfm2.5-2.6b` and `qwen3.8-27b` are
registered llama.cpp models, and `examples/routing/route_by_complexity.py` in the repo
actually executes them.

In [ ]:
ROUTE_TARGETS = {
    'quick factual question or casual small talk': 'llamacpp/lfm2.5-2.6b (local)',
    'straightforward coding or technical task': 'llamacpp/qwen3.8-27b (local)',
    'complex multi-step agentic task or deep reasoning': (
        'openai/gpt-5.6-sol | anthropic/claude-opus-5 | '
        'anthropic/claude-fable-5 | xai/grok-4.6'
    ),
    'creative writing': 'openai/gpt-5.6-terra',
    'off-topic or low-value request': 'openai/gpt-5.6-luna (or decline)',
}

traffic = [
    "What's the capital of Australia?",
    'Refactor this Python function to use dataclasses and add type hints.',
    'Plan and execute a migration of our billing system, with rollback and phased cutover.',
    'Write a short story about a lighthouse keeper who befriends a whale.',
    "What's a good pizza topping?",
]

for r in route_prompts(traffic, list(ROUTE_TARGETS), device=DEVICE):
    print(f'> {r.text}')
    print(f'  lane:  {r.best_route}  ({r.scores[0].score:.1%})')
    print(f'  model: {ROUTE_TARGETS[r.best_route]}\n')

## 8. Async

Every task has an `_async` twin — handy when routing sits in an async web handler in
front of your model call. Colab supports top-level `await`.

In [ ]:
from aibackends.tasks import route_prompts_async

t = time.perf_counter()
async_results = await route_prompts_async(traffic, list(ROUTE_TARGETS), device=DEVICE)
elapsed = time.perf_counter() - t
print(f'{len(async_results)} results in {elapsed:.2f}s without blocking the loop')

## 9. From the CLI

Same task, no Python. Routing lanes ride the `--labels` option, and the JSON output
pipes into `jq` nicely.

In [ ]:
!aibackends task route-prompt \
    --input "Can you help me debug a failing Python unit test?" \
    --labels "coding,sales,creative writing,general knowledge"

## Recap

```python
from aibackends.tasks import route_prompt, route_prompts

route_prompt(text, routes)                 # ranked RoutingResult, best_route first
route_prompt(text, routes, threshold=0.4)  # lanes below the bar are dropped
route_prompts(texts, routes)               # batch
# ...plus _async variants of both
```

Every call takes `device`, `threshold`, and `backend`.

Things worth remembering:

- Lanes are free text; renaming, adding, or removing one is just editing a list.
- Scores are a softmax over your lanes — relative preferences, not absolute
  probabilities — so tune `threshold` against your own lane set.
- The model loads once per process and device; preload it if first-request latency matters.
- Routing never touches your generative runtime; it is an independent capability backend.

Want a different router behind the same API? Register your own backend with
`register_routing_backend` and pass `backend='your-name'`.

**Links**

- Repo — https://github.com/donvito/aibackends
- Model card — https://huggingface.co/LiquidAI/LFM2.5-Encoder-350M-Prompt-Router
- Liquid AI encoders blog — https://www.liquid.ai/blog/lfm2-5-encoders